# Coachly NLU — Colab T4 Notebook (Qwen 0.5B, dataset v2)

Pipeline: **mount Drive → GPU check → deps → dataset_creator_v2 → augment → QLoRA train → eval → inference**

**Requisiti Drive:**
- `MyDrive/voice-ml-recognizer/refactor/` deve contenere:
  - `dataset_creator_v2.py`, `augment.py`, `colab_functiongemma_train.py`
  - `exercises/` (file .md degli esercizi)

**Runtime:** T4 GPU (16 GB VRAM).

In [ ]:
# ── 1) Mount Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR    = '/content/drive/MyDrive/voice-ml-recognizer'
REFACTOR    = os.path.join(REPO_DIR, 'refactor')

if not os.path.isdir(REFACTOR):
    raise FileNotFoundError(f'Cartella refactor non trovata: {REFACTOR}')

%cd {REFACTOR}
print('CWD:', os.getcwd())

In [ ]:
# ── 2) GPU check ───────────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch
assert torch.cuda.is_available(), 'GPU non disponibile! Vai su Runtime → Cambia tipo di runtime → T4'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── 3) Dipendenze ──────────────────────────────────────────────────────────────
!pip install -q -U \
    "transformers>=4.46.0" \
    "datasets>=2.20.0" \
    "accelerate>=0.33.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.43.0" \
    sentencepiece protobuf huggingface_hub

# Verifica bitsandbytes CUDA
import bitsandbytes as bnb
print(f'bitsandbytes: {bnb.__version__}')

import transformers, peft, datasets
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')
print(f'datasets:     {datasets.__version__}')

In [ ]:
# ── 4) Verifica script e directory exercises ───────────────────────────────────
import os
from pathlib import Path

SCRIPTS = [
    'dataset_creator_v2.py',
    'augment.py',
    'colab_functiongemma_train.py',
]

for s in SCRIPTS:
    p = Path(s)
    if p.exists():
        print(f'  OK  {s}')
    else:
        raise FileNotFoundError(f'Script mancante: {p.absolute()}')

ex_dir = Path('exercises')
md_files = list(ex_dir.glob('*.md')) if ex_dir.is_dir() else []
if not md_files:
    raise FileNotFoundError(f'Nessun .md in {ex_dir.absolute()}')
print(f'  OK  exercises/ ({len(md_files)} file .md)')

In [ ]:
# ── 5) Build dataset v2 (80 000 campioni) ──────────────────────────────────────
# Cambia --total per ridurre i tempi (es. 20000 per un test rapido)
!python dataset_creator_v2.py \
    --total 80000 \
    --output_dir data_v2 \
    --md_dir exercises

import json
from pathlib import Path
meta = json.loads(Path('data_v2/metadata.json').read_text(encoding='utf-8'))
print('\nSizes:', meta['sizes'])
print('Actions (all):', meta['stats']['all']['action'])
print('Langs   (all):', meta['stats']['all']['lang'])

In [ ]:
# ── 6) Augmentazione train set (factor=1.5 → +50%) ─────────────────────────────
!python augment.py \
    --input  data_v2/train.jsonl \
    --output data_v2/train_aug.jsonl \
    --factor 1.5

In [ ]:
# ── 7) QLoRA training su T4 ────────────────────────────────────────────────────
# Iperparametri ottimizzati per T4 (16 GB VRAM) + Qwen 0.5B 4-bit
BASE_MODEL  = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR  = 'output/t4_qlora'
DATA_DIR    = 'data_v2'
TRAIN_FILE  = 'train_aug.jsonl'

cmd = (
    f'python colab_functiongemma_train.py'
    f' --data_dir {DATA_DIR}'
    f' --train_file {TRAIN_FILE}'
    f' --output_dir {OUTPUT_DIR}'
    f' --base_model {BASE_MODEL}'
    f' --max_seq_len 512'
    f' --num_epochs 5'
    f' --train_batch_size 4'
    f' --eval_batch_size 4'
    f' --grad_accum 4'
    f' --lr 2e-4'
    f' --warmup_ratio 0.06'
    f' --eval_samples 200'
)
print(cmd)
!$cmd

In [ ]:
# ── 8) Metriche quick eval ─────────────────────────────────────────────────────
import json
from pathlib import Path

q = Path('output/t4_qlora/quick_eval.json')
if q.exists():
    print(json.dumps(json.loads(q.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
else:
    print('quick_eval.json non trovato — il training non è stato completato')

In [ ]:
# ── 9) Inference / test manuale ────────────────────────────────────────────────
import json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

ADAPTER_DIR = 'output/t4_qlora/adapter'
BASE_MODEL  = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE      = 'cuda'

SYSTEM = (
    'You are Coachly NLU. Convert workout speech-to-text into strict JSON.\n'
    'Return ONLY valid JSON, no markdown.\nSchema:\n'
    '{\n'
    '  "action": "ADD_EXERCISE|LOG_SET|UPDATE_SET|DELETE_EXERCISE|UNKNOWN",\n'
    '  "items": [\n'
    '    {\n'
    '      "exercise": string,\n'
    '      "sets": integer|null,\n'
    '      "reps": integer|null,\n'
    '      "weight": number|null,\n'
    '      "unit": "kg"|"lbs"|null,\n'
    '      "modifier": "to_failure"|"dropset"|"superset"|"amrap"|"pause"|null\n'
    '    }\n'
    '  ]\n'
    '}'
)

TESTS = [
    # Italiano
    'aggiungi bench press 3x10 80kg e squat 4x8 100kg e deadlift 5x5 140kg a cedimento',
    'aggiungi panca no aspetta squat 4x8 100kg e poi anche trazioni 3x6',
    'ho fatto squat 10 reps 100kg e aggiungi panca piana 3x8',
    'togli squat e deadlift',
    'aggiungi panca superset trazioni a cedimento',
    'aggiorna squat 5x5 120kg',
    'quante calorie ho bruciato',
    # English
    'add pull ups 4 sets to failure and dips 3x12 and bench press 3 sets of 8 at 80 kg dropset',
    'done bench press 8 reps 80kg then add pull ups 4x6 to failure',
    'remove deadlift and squat',
    'how many sets should i do and add bench press 3x10 80kg',
]

print(f'Loading model from {ADAPTER_DIR} on {DEVICE}...')
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
base  = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_cfg, device_map=DEVICE
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()
print('Pronto.\n')

def predict(text):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': text}],
        tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            input_ids=ids, max_new_tokens=200, do_sample=False,
            temperature=0.0, pad_token_id=tokenizer.eos_token_id
        )
    raw = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw)
    try:
        return json.JSONDecoder().raw_decode(raw)[0]
    except Exception:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {'_raw': raw}

for text in TESTS:
    result = predict(text)
    action = result.get('action', '?')
    print(f'[{action:16s}] {text}')
    print(f'  {json.dumps(result, ensure_ascii=False)}\n')

In [ ]:
# ── 10) (Opzionale) Copia adapter su Drive ─────────────────────────────────────
# L'adapter è già in refactor/output/t4_qlora/adapter/ dentro Drive,
# quindi è già persistito. Questa cella è un reminder.
import os
adapter_path = os.path.join(os.getcwd(), 'output/t4_qlora/adapter')
print(f'Adapter salvato in Drive:')
print(f'  {adapter_path}')
print()
print('Per usarlo in test_local.py:')
print('  ADAPTER_DIR = "./adapter"   # punta a output/t4_qlora/adapter')